In [ ]:
import pymc as pm, arviz as az, polars as pl

In [ ]:
df = pl.read_csv("../../Rethinking_2/End_of_chapter_problems/data/chimpanzees.csv", separator=";").drop("recipient", "trial").rename({"condition":"partner_present"})

In [ ]:
df = df.with_columns(
    treatment=pl.col("prosoc_left")+2*pl.col("partner_present"),
    actor=pl.col("actor")-1
)
df.group_by("actor", "treatment").count()

In [ ]:
with pm.Model(coords={
    "treatment": df["treatment"].unique(),
    "actor": df["actor"].unique(),
}) as base_model:
    alpha = pm.Normal("alpha", mu=0, sigma=1, dims="actor")
    beta = pm.Normal("beta", mu=0, sigma=1, dims="treatment")

    probs = pm.Deterministic("probs", pm.invlogit(alpha[df["actor"].to_numpy()] + beta[df["treatment"].to_numpy()]))

    pulled_left = pm.Bernoulli("pulled_left", logit_p=probs, observed=df["pulled_left"].to_numpy())

    # sample priors
    prior_predictive = pm.sample_prior_predictive(var_names=["pulled_left", "probs"])

    # sample å
    posterior = pm.sample(chains=4)
    posterior_predictive = pm.sample_posterior_predictive(posterior)
    pm.compute_log_likelihood(posterior)

In [ ]:
az.plot_dist(prior_predictive.prior["probs"].values.flatten())

In [ ]:
az.plot_trace(posterior, legend=True, var_names=["alpha", "beta"])